In [29]:
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

# Paths for comparing experiments (optional).
EXP_DIRS = ["200genes_full/tau12_chrombpnet_dist", "200genes_log1p/tau12_chrombpnet_dist"]
CONFIGS = ["full", "nowg", "norhog", "nowg_norhog"]

GENOTYPE_DIR = "/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/genotypes"
ANNOTATION_DIR = "/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/annotations_minmax"

GENES_LIST_FULL = "/gpfs/commons/home/vmazeeva/firvTWAS/parmigiano-expr/src/genes_list_seed_random_full.txt"
GENES_LIST_LOG1P = "/gpfs/commons/home/vmazeeva/firvTWAS/parmigiano-expr/src/genes_list_seed_random_log1p.txt"

# Run the notebook with cwd = parmigiano-expr/src so `import load_data` matches joint training.
SRC_ROOT = Path.cwd()


In [30]:
import load_data


def load_config_from_run(run_dir):
    """Load saved `config.yaml` from a single refit directory (e.g. .../full/run_1)."""
    run_dir = Path(run_dir)
    cfg_path = run_dir / "config.yaml"
    if not cfg_path.is_file():
        raise FileNotFoundError(cfg_path)
    with open(cfg_path) as f:
        return yaml.safe_load(f)


def load_config_from_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def _ensure_genes_in_config(config, src_root=None):
    """Mirror parmigiano_joint.main(): fill `genes` from gene_list file if missing."""
    if config.get("genes"):
        return config
    src_root = Path(src_root) if src_root is not None else SRC_ROOT
    gl = config.get("gene_list")
    if isinstance(gl, str):
        p = Path(gl)
        if not p.is_file():
            p = src_root / gl
        if p.is_file():
            with open(p) as f:
                config = {**config, "genes": [line.strip() for line in f if line.strip()]}
    return config


def _subset_genes(config, max_genes, random_seed=0):
    """Shrink config['genes'] for low-memory diagnostics (random subset if random_seed set, else first N)."""
    config = dict(config)
    genes = list(config.get("genes") or [])
    if max_genes is None or len(genes) <= max_genes:
        return config
    if random_seed is not None:
        rng = np.random.default_rng(random_seed)
        pick = np.sort(rng.choice(len(genes), size=max_genes, replace=False))
        genes = [genes[i] for i in pick]
    else:
        genes = genes[:max_genes]
    config["genes"] = genes
    return config


def load_diagnostic_data(
    config,
    device=None,
    split="train",
    src_root=None,
    max_genes=None,
    gene_subset_seed=0,
    annotations_only=False,
    force_cpu=True,
    skip_brr=False,
):
    """
    Rebuild tensors like joint training, with options for limited RAM (~10 GB nodes).

    Parameters
    ----------
    max_genes : int, optional
        Use only this many genes (random subset if gene_subset_seed is not None, else first N).
        Strongly reduces genotype matrix size (G is N × P variants).
    annotations_only : bool
        If True, load only Z via `load_data.load_annotations_only` — no expression TPM, no G.
        Best for `summarize_lin2_exp` and other Z-only diagnostics. Ignores `split` and BRR.
    force_cpu : bool
        If True (default), keep tensors on CPU to avoid duplicating large matrices on GPU.
    skip_brr : bool
        If True, do not load BRR results (faster, less RAM; G/Z are not filtered to BRR variants).

    Returns
    -------
    data : DataTensors or AnnotationTensors
    Z : pandas.DataFrame
    variant_ids : tuple (variant_ids_G, variant_ids_Z); G list empty when annotations_only.
    """
    config['annotation_dir'] = ANNOTATION_DIR
    config['chrombpnet_dist_only'] = False
    if force_cpu:
        device = torch.device("cpu")
    elif device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    config = _ensure_genes_in_config(dict(config), src_root=src_root)
    if max_genes is not None:
        config = _subset_genes(config, max_genes, gene_subset_seed)

    if annotations_only:
        Z, _gidx, _gnames, variant_ids_Z = load_data.load_annotations_only(config)
        Z_t = torch.as_tensor(Z.values, dtype=torch.float32, device=device)
        data = load_data.AnnotationTensors(
            Z=Z_t,
            gene_indices=_gidx,
            gene_names=_gnames,
            device=device,
        )
        return data, Z, ([], variant_ids_Z)

    X, Y, train_idx, test_idx = load_data.load_residualized_covariates(config, device)
    G, Z, variant_ids_G, variant_ids_Z = load_data.load_genes(config)

    load_cfg = dict(config)
    if skip_brr:
        load_cfg["use_brr"] = False
        load_cfg["brr_results_dir"] = None
    use_brr = load_cfg.get("use_brr", True) and load_cfg.get("brr_results_dir")
    if use_brr:
        brr = load_data.load_brr_results(load_cfg)
        brr_betas = brr.get("betas") if brr else None
        brr_alphas = brr.get("alphas") if brr else None
    else:
        brr_betas = brr_alphas = None

    if config.get("train_test", False) and split == "test":
        test_sample_ids = X.iloc[test_idx].index
        X_test = X.iloc[test_idx]
        Y_test = {gene: expr[test_idx] for gene, expr in Y.items()}
        G_test = G.loc[test_sample_ids]
        data = load_data.DataTensors.from_pandas(
            G_test, Z, X_test, Y_test, brr_betas, brr_alphas, device, load_cfg
        )
    else:
        if config.get("train_test", False):
            train_sample_ids = X.iloc[train_idx].index
            X_train = X.iloc[train_idx]
            Y_train = {gene: expr[train_idx] for gene, expr in Y.items()}
            G_train = G.loc[train_sample_ids]
        else:
            X_train, Y_train, G_train = X, Y, G
        data = load_data.DataTensors.from_pandas(
            G_train, Z, X_train, Y_train, brr_betas, brr_alphas, device, load_cfg
        )

    return data, Z[['chromBPnet', 'dist_to_TSS']], (variant_ids_G, variant_ids_Z)


def read_tau_T_csv(path):
    """Read final `tau_T.csv` (Annotation, Tau1, Filter Threshold, Tau2)."""
    return pd.read_csv(path)


def tau_T_to_torch(tau_T_df, device=None):
    """
    Parse `tau_T.csv` into tensors. Nonlinear: Tau1, Tau2. Linear: Tau only.
    Annotation order matches `Z` columns from `load_diagnostic_data`.
    """
    if device is None:
        device = torch.device("cpu")
    df = tau_T_df
    ann_col = "Annotation" if "Annotation" in df.columns else df.columns[0]
    thr_col = "Filter Threshold" if "Filter Threshold" in df.columns else "Filter_Threshold"
    thr = float(df[thr_col].iloc[0])
    ann = df[ann_col].tolist()
    if "Tau2" in df.columns and "Tau1" in df.columns:
        tau1 = torch.tensor(df["Tau1"].values, dtype=torch.float32, device=device)
        tau2 = torch.tensor(df["Tau2"].values, dtype=torch.float32, device=device)
        return {
            "annotations": ann,
            "tau1": tau1,
            "tau2": tau2,
            "threshold": thr,
            "mode": "nonlinear",
        }
    if "Tau" in df.columns:
        tau = torch.tensor(df["Tau"].values, dtype=torch.float32, device=device)
        return {
            "annotations": ann,
            "tau": tau,
            "threshold": thr,
            "mode": "linear",
        }
    raise ValueError("tau_T.csv needs Tau (linear) or Tau1+Tau2 (nonlinear).")


def load_tau_history_runs(exp_root, max_runs=None):
    """
    Load every `tau_history.csv` under exp_root/run_*/ (sorted by run index).

    Each DataFrame gets a `run` column (integer). Nonlinear: columns epoch, Tau1_*, Tau2_*.
    """
    exp_root = Path(exp_root)
    pat = re.compile(r"^run_(\d+)$")
    runs = []
    for name in os.listdir(exp_root):
        m = pat.match(name)
        if m:
            runs.append((int(m.group(1)), name))
    runs.sort(key=lambda x: x[0])
    if max_runs is not None:
        runs = runs[:max_runs]
    out = []
    for _, dirname in runs:
        csv_path = exp_root / dirname / "tau_history.csv"
        if not csv_path.is_file():
            continue
        df = pd.read_csv(csv_path)
        df["run"] = int(dirname.split("_")[1])
        out.append(df)
    return out


def mean_tau_trajectory_by_run_groups(run_dfs, group_size=10, value_cols=None):
    """
    Split ordered runs into bins of `group_size`; within each bin, average trajectories epoch-wise.

    Returns a list of dicts with keys: run_range (low, high), epoch, mean (DataFrame), sem (DataFrame).
    """
    if not run_dfs:
        return []
    if value_cols is None:
        value_cols = [
            c
            for c in run_dfs[0].columns
            if c not in ("epoch", "run")
        ]
    groups = []
    for i in range(0, len(run_dfs), group_size):
        chunk = run_dfs[i : i + group_size]
        min_len = min(len(d) for d in chunk)
        epochs = chunk[0]["epoch"].values[:min_len]
        arr = np.stack([d[value_cols].values[:min_len].astype(float) for d in chunk], axis=0)
        mean_vals = arr.mean(axis=0)
        if arr.shape[0] <= 1:
            sem_vals = np.zeros_like(mean_vals)
        else:
            sem_vals = arr.std(axis=0, ddof=1) / np.sqrt(arr.shape[0])
        r0, r1 = int(chunk[0]["run"].iloc[0]), int(chunk[-1]["run"].iloc[-1])
        groups.append(
            {
                "run_range": (r0, r1),
                "epoch": epochs,
                "mean": pd.DataFrame(mean_vals, columns=value_cols),
                "sem": pd.DataFrame(sem_vals, columns=value_cols),
            }
        )
    return groups


def plot_tau_history_groups(
    exp_root,
    group_size=10,
    max_runs=None,
    figsize=(12, 5),
    sem_alpha=0.2,
    title=None,
):
    """
    For each block of `group_size` consecutive runs, plot mean ± SEM vs epoch.

    Nonlinear: one **row per annotation** (e.g. chrombpnet vs dist_to_TSS), columns **Tau1 | Tau2**.
    Linear: single panel of tau columns.
    """
    run_dfs = load_tau_history_runs(exp_root, max_runs=max_runs)
    if not run_dfs:
        raise FileNotFoundError(f"No tau_history.csv under {exp_root}/run_*")

    cols = [c for c in run_dfs[0].columns if c not in ("epoch", "run")]
    tau1_cols = [c for c in cols if c.startswith("Tau1_")]
    tau2_cols = [c for c in cols if c.startswith("Tau2_")]

    if tau1_cols and tau2_cols:
        ann_from_t1 = {c.split("_", 1)[1] for c in tau1_cols}
        ann_from_t2 = {c.split("_", 1)[1] for c in tau2_cols}
        ann_names = sorted(ann_from_t1 & ann_from_t2)
        if not ann_names:
            ann_names = sorted(ann_from_t1 | ann_from_t2)

        n_rows = len(ann_names)
        h = max(3.0, 2.8 * n_rows)
        w = figsize[0] if figsize[0] >= 10 else 12
        fig, axes = plt.subplots(
            n_rows,
            2,
            figsize=(w, h),
            sharex=True,
            constrained_layout=True,
            squeeze=False,
        )
        for row, ann in enumerate(ann_names):
            c1 = f"Tau1_{ann}"
            c2 = f"Tau2_{ann}"
            pair = [c for c in (c1, c2) if c in cols]
            if not pair:
                continue
            groups = mean_tau_trajectory_by_run_groups(run_dfs, group_size, value_cols=pair)
            ax1, ax2 = axes[row, 0], axes[row, 1]
            for g in groups:
                e = g["epoch"]
                label_prefix = f"runs {g['run_range'][0]}–{g['run_range'][1]}"
                if c1 in pair:
                    m = g["mean"][c1].values
                    s = g["sem"][c1].values
                    ax1.plot(e, m, label=label_prefix)
                    ax1.fill_between(e, m - s, m + s, alpha=sem_alpha)
                if c2 in pair:
                    m = g["mean"][c2].values
                    s = g["sem"][c2].values
                    ax2.plot(e, m, label=label_prefix)
                    ax2.fill_between(e, m - s, m + s, alpha=sem_alpha)
            ax1.set_ylabel("weight")
            ax2.set_ylabel("weight")
            ax1.set_title(f"{ann} — Tau1")
            ax2.set_title(f"{ann} — Tau2")
            ax1.legend(fontsize=7, loc="best")
            ax2.legend(fontsize=7, loc="best")
            if row == n_rows - 1:
                ax1.set_xlabel("epoch")
                ax2.set_xlabel("epoch")
        if title:
            fig.suptitle(title)
    else:
        fig, ax = plt.subplots(figsize=(figsize[0], figsize[1] / 2), constrained_layout=True)
        groups = mean_tau_trajectory_by_run_groups(run_dfs, group_size, value_cols=cols)
        for g in groups:
            e = g["epoch"]
            label_prefix = f"runs {g['run_range'][0]}–{g['run_range'][1]}"
            for col in cols:
                m = g["mean"][col].values
                s = g["sem"][col].values
                ax.plot(e, m, label=f"{label_prefix} | {col}")
                ax.fill_between(e, m - s, m + s, alpha=sem_alpha)
        ax.set_xlabel("epoch")
        ax.set_ylabel("tau")
        ax.legend(fontsize=7, loc="best")
        if title:
            ax.set_title(title)
    return fig


# Example (uncomment; requires cwd = src and data paths valid):
# RUN_DIR = SRC_ROOT / "200genes_full/tau12_chrombpnet_dist/full/run_1"
# cfg = load_config_from_run(RUN_DIR)
# data, Z_df, vids = load_diagnostic_data(cfg, split="train")
# tau_df = read_tau_T_csv(RUN_DIR / "tau_T.csv")
# tau_tensors = tau_T_to_torch(tau_df, device=data.device)
# fig = plot_tau_history_groups(SRC_ROOT / "200genes_full/tau12_chrombpnet_dist/full", group_size=10)
# plt.show()


In [31]:
import torch

def summarize_lin2_exp(data, tau2, genes=None):
    """tau2: (Q,) same ordering as Z columns. Matches model: exp(Z·τ₂) with no clamp."""
    rows = []
    tau2 = tau2.reshape(-1)
    for gene_name in (genes or data.gene_names):
        _, Z_gene, _ = data.get_gene_data(gene_name)
        lin2 = (Z_gene @ tau2).detach().float().cpu()
        er = torch.exp(lin2)
        rows.append({
            "gene": gene_name,
            "lin2_max": float(lin2.max()),
            "lin2_p99": float(torch.quantile(lin2, 0.99)),
            "exp_max": float(er.max()),
        })
    return rows

# After load_diagnostic_data + tau_T_to_torch:
#   r = summarize_lin2_exp(data, tau_tensors["tau2"])
#   pd.DataFrame(r).describe()

In [32]:
def _tau_hist_columns_for_Z(Z_df):
    """Map Z columns to tau_history column names (Tau1_*, Tau2_*)."""
    ann = list(Z_df.columns)
    t1 = [f"Tau1_{a}" for a in ann]
    t2 = [f"Tau2_{a}" for a in ann]
    return ann, t1, t2


def compute_Z_dot_tau_trajectory(Z_df, tau_hist_df):
    """
    For each epoch in `tau_history.csv`, compute Z·τ₁ and Z·τ₂ (all variants pooled),
    per-annotation marginals, and **λ = (Z·τ₁)·exp(Z·τ₂)** (no MAF; matches `annotation_lambda`).
    """
    Zmat = np.asarray(Z_df.values, dtype=np.float64)
    ann, cols1, cols2 = _tau_hist_columns_for_Z(Z_df)
    missing = [c for c in cols1 + cols2 if c not in tau_hist_df.columns]
    if missing:
        raise ValueError(f"tau_history missing columns {missing}. Have: {list(tau_hist_df.columns)}")

    out_rows = []
    for _, row in tau_hist_df.iterrows():
        tau1 = row[cols1].to_numpy(dtype=np.float64)
        tau2 = row[cols2].to_numpy(dtype=np.float64)
        lin1 = Zmat @ tau1
        lin2 = Zmat @ tau2
        with np.errstate(over="ignore"):
            exp2 = np.exp(lin2)
        lam = lin1 * exp2

        rec = {
            "epoch": int(row["epoch"]),
            "lin1_max": float(np.nanmax(lin1)),
            "lin1_p99": float(np.percentile(lin1, 99)),
            "lin1_p50": float(np.percentile(lin1, 50)),
            "lin1_min": float(np.nanmin(lin1)),
            "lin2_max": float(np.nanmax(lin2)),
            "lin2_p99": float(np.percentile(lin2, 99)),
            "lin2_p50": float(np.percentile(lin2, 50)),
            "lin2_min": float(np.nanmin(lin2)),
            "exp_lin2_max": float(np.nanmax(exp2)) if np.any(np.isfinite(exp2)) else float("inf"),
            "exp_lin2_p99": float(np.percentile(exp2[np.isfinite(exp2)], 99))
            if np.any(np.isfinite(exp2))
            else float("nan"),
            "lam_absmax": float(np.nanmax(np.abs(lam[np.isfinite(lam)])))
            if np.any(np.isfinite(lam))
            else float("inf"),
            "lam_p99": float(np.percentile(np.abs(lam[np.isfinite(lam)]), 99))
            if np.any(np.isfinite(lam))
            else float("nan"),
        }
        for j, a in enumerate(ann):
            p1 = Zmat[:, j] * tau1[j]
            p2 = Zmat[:, j] * tau2[j]
            rec[f"Ztau1_{a}_absmax"] = float(np.nanmax(np.abs(p1)))
            rec[f"Ztau1_{a}_p99"] = float(np.percentile(np.abs(p1), 99))
            rec[f"Ztau2_{a}_absmax"] = float(np.nanmax(np.abs(p2)))
            rec[f"Ztau2_{a}_p99"] = float(np.percentile(np.abs(p2), 99))
        out_rows.append(rec)
    return pd.DataFrame(out_rows)


def plot_Z_dot_tau_over_epochs(
    Z_df,
    tau_hist_df,
    title=None,
    figsize=None,
):
    """
    Plot Z·τ₁, Z·τ₂, |λ| = |(Z·τ₁)·exp(Z·τ₂)| (no MAF; no exp clamp), exp(Z·τ₂), and per-annotation marginals.
    """
    stats = compute_Z_dot_tau_trajectory(Z_df, tau_hist_df)
    ep = stats["epoch"].values
    ann, _, _ = _tau_hist_columns_for_Z(Z_df)
    n_ann = len(ann)
    n_top = 4
    if figsize is None:
        figsize = (12, 2.6 * (n_top + n_ann))

    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(n_top + n_ann, 2)

    ax = fig.add_subplot(gs[0, :])
    ax.plot(ep, stats["lin1_max"], label="max", color="C0")
    ax.plot(ep, stats["lin1_p99"], label="p99", color="C1")
    ax.plot(ep, stats["lin1_p50"], label="median", color="C2")
    ax.plot(ep, stats["lin1_min"], label="min", color="C3", alpha=0.7)
    ax.set_ylabel("Z·τ₁")
    ax.set_title("Pooled over variants: linear term for τ₁ path")
    ax.legend(fontsize=8, ncol=4, loc="upper right")
    ax.set_xlabel("epoch")

    ax = fig.add_subplot(gs[1, :])
    ax.plot(ep, stats["lin2_max"], label="max", color="C0")
    ax.plot(ep, stats["lin2_p99"], label="p99", color="C1")
    ax.plot(ep, stats["lin2_p50"], label="median", color="C2")
    ax.plot(ep, stats["lin2_min"], label="min", color="C3", alpha=0.7)
    ax.set_ylabel("Z·τ₂")
    ax.set_title("Pooled over variants: input to exp(model uses exp(Z·τ₂), no clamp)")
    ax.legend(fontsize=8, ncol=4, loc="upper right")
    ax.set_xlabel("epoch")

    ax = fig.add_subplot(gs[2, :])
    ax.plot(ep, stats["lam_absmax"], label="|λ| max", color="C0")
    ax.plot(ep, stats["lam_p99"], label="|λ| p99", color="C1")
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_ylabel("|(Z·τ₁)·exp(Z·τ₂)|  (no MAF)")
    ax.set_title("λ scale (matches annotation_lambda)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlabel("epoch")

    ax = fig.add_subplot(gs[3, :])
    ax.plot(ep, stats["exp_lin2_max"], label="max exp(Z·τ₂)", color="darkred")
    ax.plot(ep, stats["exp_lin2_p99"], label="p99 exp(Z·τ₂)", color="coral")
    ax.set_yscale("symlog", linthresh=1.0)
    ax.set_ylabel("exp(Z·τ₂)")
    ax.set_title("exp(Z·τ₂) (symlog)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlabel("epoch")

    for i, a in enumerate(ann):
        ax1 = fig.add_subplot(gs[n_top + i, 0])
        ax1.plot(ep, stats[f"Ztau1_{a}_absmax"], label="max |Z[:,a]·τ₁[a]|", color="C0")
        ax1.plot(ep, stats[f"Ztau1_{a}_p99"], label="p99 |.|", color="C1")
        ax1.set_title(f"{a}: marginal τ₁ term (per-variant)")
        ax1.set_ylabel("|Zₐ×τ₁ₐ|")
        ax1.legend(fontsize=7)
        if i == n_ann - 1:
            ax1.set_xlabel("epoch")

        ax2 = fig.add_subplot(gs[n_top + i, 1])
        ax2.plot(ep, stats[f"Ztau2_{a}_absmax"], label="max |Z[:,a]·τ₂[a]|", color="C0")
        ax2.plot(ep, stats[f"Ztau2_{a}_p99"], label="p99 |.|", color="C1")
        ax2.set_title(f"{a}: marginal τ₂ term (per-variant)")
        ax2.set_ylabel("|Zₐ×τ₂ₐ|")
        ax2.legend(fontsize=7)
        if i == n_ann - 1:
            ax2.set_xlabel("epoch")

    if title:
        fig.suptitle(title)
    return fig, stats

### Annotation distributions (`chrombpnet_dist_only`)

After loading with **`load_diagnostic_data`** (or full `load_genes`), `Z_df` has **`chrombpnet`** (mean of raw chrombpnet columns, then **z-scored** across variants) and **`dist_to_TSS`** (**log(|distance|)**, not z-scored)—same as **`load_data.load_genes`**.

Use **`plot_chrombpnet_dist_distributions(Z_df, ...)`** in the next cell to compare histograms (density-normalized).

In [33]:
def plot_chrombpnet_dist_distributions(
    Z_df,
    title=None,
    bins=80,
):
    """
    Histograms for training annotations under chrombpnet_dist_only preprocessing.

    Expects columns ``chrombpnet`` (z-scored) and ``dist_to_TSS`` (log |distance|, raw scale).
    """
    need = ("chrombpnet", "dist_to_TSS")
    missing = [c for c in need if c not in Z_df.columns]
    if missing:
        raise ValueError(
            f"Z_df missing columns {missing}. For chrombpnet_dist_only, load with that flag in config "
            f"(current columns: {list(Z_df.columns)[:20]}{'...' if Z_df.shape[1] > 20 else ''})"
        )

    subtitles = (
        "chrombpnet (z-scored, global mean 0 / std 1 over variants)",
        "dist_to_TSS (log |distance|, not z-scored)",
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)

    for ax, col, st in zip(axes, need, subtitles):
        v = Z_df[col].dropna().to_numpy(dtype=float)
        if v.size == 0:
            ax.set_title(f"{col}: no data")
            continue
        ax.hist(
            v,
            bins=bins,
            density=True,
            color="steelblue",
            alpha=0.78,
            edgecolor="white",
            linewidth=0.35,
        )
        mu = float(np.nanmean(v))
        ax.axvline(mu, color="darkred", linestyle="--", linewidth=1.0, label=f"mean={mu:.4g}")
        ax.set_xlabel(col)
        ax.set_ylabel("density")
        ax.set_title(st, fontsize=10)
        ax.legend(fontsize=8, loc="best")

    if title:
        fig.suptitle(title, fontsize=11)
    return fig


# Requires cfg with chrombpnet_dist_only=True and a valid gene list (e.g. from load_config_from_run)
# _, Z_anno, _ = load_diagnostic_data(cfg, annotations_only=True, force_cpu=True, max_genes=None)
# plot_chrombpnet_dist_distributions(Z_anno, title="Training Z (annotations_only, full gene list)")
# plt.show()

### Quick start

1. **Kernel / working directory:** open this notebook from `parmigiano-expr/src` (or `os.chdir` there) so `import load_data` works and relative paths in saved `config.yaml` match training.
2. Point `RUN_DIR` at any `.../run_k` folder that contains `config.yaml`, `tau_T.csv`, and `tau_history.csv`.
3. `plot_tau_history_groups(exp_parent, group_size=10)` expects `exp_parent` = directory that **contains** `run_1`, `run_2`, … (e.g. `.../tau12_chrombpnet_dist/full`).

### Memory (~10 GB nodes)

- **Z-only diagnostics** (`summarize_lin2_exp`, etc.): use `load_diagnostic_data(..., annotations_only=True)`. Skips the full TPM matrix and the genotype matrix `G` (the main RAM user).
- **Full `DataTensors` but fewer variants:** pass `max_genes=30` (tune up/down) and `skip_brr=True` to avoid loading all BRR tables and to skip BRR variant filtering.
- **`force_cpu=True`** (default) avoids a second full copy on GPU.
- Order-of-magnitude: full 200-gene train `G` is often **several GB** in float32; `annotations_only` keeps only **Z** (variants × few annotations).

**Z·τ vs epoch:** `plot_Z_dot_tau_over_epochs(Z_df, tau_hist_df)` — pooled **Z·τ₁**, **Z·τ₂**, **|λ| = |(Z·τ₁)·exp(Z·τ₂)|** (no MAF; matches `annotation_lambda`, no clamp), **exp(Z·τ₂)** symlog, per-annotation marginals.

In [ ]:
# --- edit these ---
RUN_DIR = SRC_ROOT / "200genes_full/tau12_chrombpnet_dist/full/run_1"
EXP_PARENT = SRC_ROOT / "200genes_full/tau12_chrombpnet_dist/full"  # all run_* live here

cfg = load_config_from_run(RUN_DIR)
# Low memory: annotations only (no G, no expression). Optional: max_genes=40
data, Z_df, _ = load_diagnostic_data(
    cfg,
    annotations_only=True,
    max_genes=None,
    force_cpu=True,
)
tau_df = read_tau_T_csv(RUN_DIR / "tau_T.csv")
tau_tensors = tau_T_to_torch(tau_df, device=data.device)
print("Z columns:", list(Z_df.columns))
print(tau_df)

if {"chrombpnet", "dist_to_TSS"}.issubset(Z_df.columns):
    fig_z = plot_chrombpnet_dist_distributions(
        Z_df,
        title=f"Annotation distributions (as in load_genes) — {EXP_PARENT}",
    )
    plt.show()

if tau_tensors.get("mode") == "nonlinear":
    r = summarize_lin2_exp(data, tau_tensors["tau2"])
    print(pd.DataFrame(r).describe())

fig = plot_tau_history_groups(EXP_PARENT, group_size=10, title=str(EXP_PARENT.name))
plt.show()

# Z·τ₁, Z·τ₂ and per-annotation marginals vs epoch (same run's tau_history + Z_df)
tau_hist = pd.read_csv(RUN_DIR / "tau_history.csv")
fig2, zstats = plot_Z_dot_tau_over_epochs(
    Z_df,
    tau_hist,
    title=f"Z·τ trajectory — {RUN_DIR.name}",
)
plt.show()

100%|██████████| 200/200 [00:01<00:00, 143.39it/s]


Annotation-only matrix: 216146 variants × 28 features
Annotation matrix (only): 17 annotations per variant
Z columns: ['lof', 'missense', 'alphamissense', 'splice', 'dist_to_TSS', 'ABC', 'conservation', 'roadmap', 'pathogenicity', 'maf', 'promoter_4000', 'intron', 'cds', 'UTR', 'chromBPnet', 'TF_delta_min', 'TF_delta_max']
    Annotation      Tau1  Filter Threshold      Tau2
0   chrombpnet  0.823255               0.0  0.086552
1  dist_to_TSS  0.176745               0.0  0.913448


RuntimeError: size mismatch, got input (636), mat (636x17), vec (2)